# DarwinBot: does a GPU speed up training?

Measures episodes per second on the CPU (MuJoCo on every core) and on the GPU
(MJX, batched) for the same body, with batches of 16 to 1024 controllers. A
CMA-ES population is 16 to 64 controllers today; the GPU pays off only if it
wins at those sizes, or if a larger population is worth using.

**Settings**: Accelerator *GPU T4 x2* or *GPU P100*, Internet *On*. Takes about
20 minutes.

In [ ]:
# Clone DarwinBot into /tmp (not saved with the notebook output) and install it.
# While the repository is private, add a Kaggle secret named GITHUB_TOKEN
# (Add-ons > Secrets) holding a GitHub token with read access to it.
import os, subprocess
REPO = "mpena2099/darwinbot"
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
subprocess.run(["git", "clone", "-q", "--depth", "1", url, "/tmp/darwinbot"], check=True)
os.chdir("/tmp/darwinbot")
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("cores:", os.cpu_count())

In [ ]:
# MJX and a CUDA build of JAX
subprocess.run(["pip", "install", "-q", "mujoco-mjx"], check=True)
import jax
if jax.default_backend() != "gpu":
    subprocess.run(["pip", "install", "-q", "-U", "jax[cuda12]"], check=True)
    print("JAX was reinstalled with CUDA: restart the session and run again")
else:
    print(jax.devices())

In [ ]:
def bench(*extra):
    subprocess.run(["python", "tools/kaggle/bench_gpu.py", *extra])

bench("--body", "quadruped")
bench("--body", "hexapod")
bench("--body", "wheeled")
bench("--body", "quadruped", "--terrain", "rough", "--sizes", "64,256")

Reading the table: `episodes/s` on the GPU against the CPU line. The
`compile s` column is paid once for every new body, so add it to the cost of
one CMA-ES run (120 to 1200 episodes per body) before comparing.